## 0 · Goal & config
TODO: goal, model, seed, paths.

In [1]:
# Runpod only: skip this cell when running locally.
%cd /workspace/LLM-fine-tuning/Training

/workspace/LLM-fine-tuning/Training


### Set device

In [6]:
import torch

device = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if torch.backends.mps.is_available()
                      else "cpu")

print("device-name", device);                      

device-name cuda


## 1 · Load model
Load pretrained GPT-2 and move it to the device.

In [3]:
from reference.configs import gpt2_BASE_CONFIG, gpt2_model_configs

CHOOSE_MODEL = "gpt2-medium (355M)"

# ** unpacks dictionary entries: {**a, **b} combines them; b wins on duplicate keys.
model_config = {**gpt2_BASE_CONFIG, **gpt2_model_configs[CHOOSE_MODEL]}


In [4]:
%%capture
from reference.model_io import download_weights
from reference.models.gpt2 import GPT2Model, load_weights_into_gpt

# download_weights() defaults to 124M
w = download_weights("openai-community/gpt2-medium")
model = GPT2Model(model_config)
load_weights_into_gpt(model, w)

model = model.to(device)

## 2 · Data + DataLoader  ← differs
Format prompts, split, dataset, collate (see masking note), loaders.

In [5]:
from reference.data import load_instruction_data

data = load_instruction_data()
print("Instruction Data Keys: ", data[0].keys())
data[:1]

Instruction Data Keys:  dict_keys(['instruction', 'input', 'output'])


[{'instruction': 'Evaluate the following phrase by transforming it into the spelling given.',
  'input': 'freind --> friend',
  'output': 'The spelling of the given phrase "freind" is incorrect, the correct spelling is "friend".'}]

### Prompt template

In [ ]:
def format_input(entry):

  instruction_text = (
      f"Below is an instruction that describes a task. "
      f"Write a response that appropriately completes the request."
      f"\n\n### Instruction: \n{entry['instruction']}"
  )

  input_text = (
  f"\n\n### Input:\n{entry['input']}" if entry['input'] else ""
  )

  return instruction_text + input_text

model_input = format_input(data[50])
detailed_response = f"\n\n### Response:\n{data[50]['output']}"
print(model_input + detailed_response)

### Split data: train 0.85, validation 0.05, test 0.10

In [ ]:
train_portion = int(0.85 * len(data))
test_portion  = int(0.10 * len(data))
val_portion   = int(len(data) - train_portion - test_portion)

train_data = data[:train_portion]
test_data  = data[train_portion:train_portion+test_portion]
val_data   = data[train_portion+test_portion:]

### Dataset

In [ ]:
from torch.utils.data import Dataset

class InstructionDataset(Dataset):

    def __init__(self, data, tokenizer):
        self.data = data
        self.encoded_texts = []
        for entry in data:
            formatted_input    = format_input(entry)
            formatted_response = f"\n\n### Response:\n{entry['output']}"
            combined_text = formatted_input + formatted_response
            encoded_text = tokenizer.encode(combined_text)
            self.encoded_texts.append(encoded_text)


    def __getitem__(self, index):
        return self.encoded_texts[index]
    
    def __len__(self):
        return len(self.data)


### Custom Collate Function

In [ ]:
import torch

def custom_collate_fn(batch, pad_token_id, ignore_index = -100, device = "cpu", allowed_max_length = None):

    batch_max_length = max(len(item)+ 1 for item in batch)
    inputs_lst, targets_lst = [], []

    for item in batch:
        new_item = item.copy()
        new_item += [pad_token_id]

        padded = new_item + [pad_token_id] * (batch_max_length - len(new_item))
        inputs  = torch.tensor(padded[:-1])
        targets = torch.tensor(padded[1:])

        mask    = targets == pad_token_id
        indices = torch.nonzero(mask).squeeze()
        if indices.numel() > 1:
            targets[indices[1:]] = ignore_index

        if allowed_max_length is not None:
            inputs  = inputs[:allowed_max_length]
            targets = targets[:allowed_max_length]

        inputs_lst.append(inputs)
        targets_lst.append(targets)


    inputs_tensor  = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device)

    return inputs_tensor, targets_tensor

> **Masking note:** `custom_collate_fn` replaces only *padding* tokens with `ignore_index` (-100). Prompt/instruction tokens are **not** masked, so they **currently contribute to the loss** along with the response. To train on the response only, also mask the prompt-token positions.

In [ ]:
from functools import partial

import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")
pad_token_id = tokenizer.encode("<|endoftext|>", allowed_special={"<|endoftext|>"})


customized_colate_fn = partial(custom_collate_fn,
                               device=device,
                               allowed_max_length=1024,
                               pad_token_id=pad_token_id[0]) 

# error if pad_token_id = [50256] instead of pad_token_id = 50256 since function already wraps it in brackets.

### DataLoader

In [ ]:
from torch.utils.data import DataLoader

num_workers = 0
batch_size  = 8
train_dataset = InstructionDataset(train_data, tokenizer)
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    collate_fn=customized_colate_fn,
    shuffle=True,
    drop_last=True,
    num_workers=num_workers
    )

# Inspect a single example batch (not every batch)
inputs, targets = next(iter(train_loader))
print(inputs.shape, targets.shape)

In [ ]:
val_dataset = InstructionDataset(val_data, tokenizer)
val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=batch_size,
    collate_fn=customized_colate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
    )

print(len(val_loader))

In [ ]:
test_dataset = InstructionDataset(test_data, tokenizer)
test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=batch_size,
    collate_fn=customized_colate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
    )

print(len(test_loader))

## 3 · Loss / objective  ← differs
TODO: next-token cross-entropy; prompt tokens currently count (see masking note).

## 4 · Generate / predict
TODO: generate a response; capture a baseline before training.

## 5 · Train  ← differs
TODO: fine-tune on instruction data. Keep the loop visible.

## 6 · Save
TODO: extract and save responses → runs/instruction/.

## 7 · Evaluate & reflect  ← differs
TODO: score responses (optionally local Qwen); compare to baseline.